In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION — Set random seed to 42 for absolute reproducibility
# ============================================================
INPUT_FILE  = "C:/Users/Puni/Desktop/Thesis/Dataset/clinical_trials_with_targets.csv" # From Notebook 2
OUTPUT_FILE = "C:/Users/Puni/Desktop/Thesis/Dataset/clinical_trials_ml_ready.csv"      # ML ready output
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

# Load dataset using 'NCT Number' as index to align with previous notebooks
df = pd.read_csv(INPUT_FILE, index_col="NCT Number")
print(f"Dataset loaded successfully: {df.shape[0]:,} rows × {df.shape[1]} columns")

# ------------------------------------------------------------
# FEATURE 1: Broadened Therapeutic Area Mapping (Fixes Issue 2)
# ------------------------------------------------------------
def get_therapeutic_area(condition_str):
    if pd.isna(condition_str):
        return "OTHER"
    c = str(condition_str).lower()
    
    # Check smaller/specific categories first to prevent dominant ONCOLOGY overlap
    if any(k in c for k in ['hiv', 'infection', 'bacterial', 'viral', 'covid', 'hepatitis', 'tuberculosis', 'sepsis', 'influenza', 'vaccine', 'pneumonia']):
        return "INFECTIOUS DISEASE"
    if any(k in c for k in ['alzheimer', 'parkinson', 'dementia', 'epilepsy', 'stroke', 'multiple sclerosis', 'neurolog', 'neuropath', 'cognitive', 'brain', 'migraine', 'cns', 'seizure']):
        return "NEUROLOGY"
    if any(k in c for k in ['depression', 'anxiety', 'schizophrenia', 'bipolar', 'psychiatr', 'autism', 'adhd', 'ptsd', 'mood', 'insomnia']):
        return "PSYCHIATRY"
    if any(k in c for k in ['heart', 'cardiac', 'coronary', 'hypertension', 'atrial', 'myocardial', 'stroke', 'vascular', 'artery', 'thrombosis', 'embolism']):
        return "CARDIOVASCULAR"
    if any(k in c for k in ['diabetes', 'insulin', 'glucose', 'obesity', 'lipid', 'metabolic', 'thyroid', 'cholesterol']):
        return "METABOLIC"
    if any(k in c for k in ['asthma', 'pulmonary', 'copd', 'bronch', 'respiratory', 'lung chronic', 'cystic fibrosis']):
        return "RESPIRATORY"
    if any(k in c for k in ['arthritis', 'lupus', 'autoimmune', 'rheumat', 'immunology', 'psoriasis', 'crohn', 'colitis', 'inflammation']):
        return "IMMUNOLOGY"
    if any(k in c for k in ['cancer', 'carcinoma', 'lymphoma', 'leukemia', 'tumor', 'neoplasm', 'glioma', 'sarcoma', 'myeloma', 'melanoma', 'oncolog', 'malignant']):
        return "ONCOLOGY"
        
    return "OTHER"

print("Mapping Feature 1: Therapeutic Area (Broadened)...")
df["therapeutic_area"] = df["Conditions"].apply(get_therapeutic_area)
print("Therapeutic Area Distribution Profile:")
print(df["therapeutic_area"].value_counts(normalize=True) * 100)

# ------------------------------------------------------------
# FEATURE 2: Sponsor Type Mapping via Sponsor
# ------------------------------------------------------------
def get_sponsor_type(sponsor_str):
    if pd.isna(sponsor_str):
        return "INDUSTRY"
    s = str(sponsor_str).lower()
    keywords = ['university', 'hospital', 'institute', 'college', 'national', 'foundation', 'agency', 'center', 'centre']
    if any(k in s for k in keywords):
        return "ACADEMIC_GOVT"
    return "INDUSTRY"

print("\nMapping Feature 2: Sponsor Type...")
df["sponsor_type"] = df["Sponsor"].apply(get_sponsor_type)

# ============================================================
# METADATA RISK ENGINE FOR TARGET DE-COUPLING (Fixes Issue 1)
# Deriving baseline clinical weights from study design metadata columns ONLY
# ============================================================
phase_complexity = {"PHASE1": 1.0, "EARLY_PHASE1": 0.8, "PHASE2": 1.5, "PHASE3": 2.5, "PHASE4": 1.2, "NOT_REPORTED": 1.3}
interv_complexity = {"BIOLOGICAL": 1.4, "DRUG": 1.2, "DEVICE": 1.1, "BEHAVIORAL": 0.6, "RADIATION": 1.1, "PROCEDURE": 0.9, "OTHER": 1.0}

def get_meta_complexity_factor(row):
    p_fact = phase_complexity.get(str(row["Phases"]).upper().strip(), 1.3)
    i_fact = interv_complexity.get(str(row["Intervention Type"]).upper().strip(), 1.0)
    dur_years = max(0.5, row["Trial Duration (days)"] / 365.25)
    dur_fact = 0.8 if dur_years < 1 else (1.0 if dur_years <= 2 else (1.3 if dur_years <= 4 else 1.6))
    site_fact = 1.4 if row["Number of Sites"] > 20 else (1.1 if row["Number of Sites"] > 5 else 0.8)
    return p_fact * i_fact * dur_fact * site_fact

# Calculate structural baseline weight profile per trial row
meta_weights = df.apply(get_meta_complexity_factor, axis=1)

# ------------------------------------------------------------
# FEATURE 3: Protocol Amendment Count (Poisson Distributed)
# ------------------------------------------------------------
base_lambda_phase = {"PHASE1": 1.2, "PHASE2": 2.0, "PHASE3": 3.5, "PHASE4": 1.8, "NOT_REPORTED": 1.8, "EARLY_PHASE1": 1.0}

def get_amendments(row):
    phase = str(row["Phases"]).upper().strip()
    base_lam = base_lambda_phase.get(phase, 1.8)
    
    duration_days = row["Trial Duration (days)"]
    if duration_days < 365:
        mult = 0.7
    elif duration_days <= 730:
        mult = 1.0
    elif duration_days <= 1460:
        mult = 1.4
    else:
        mult = 1.8
    
    # Modulate via metadata weight profile rather than raw target percentiles
    lam = base_lam * mult * (0.8 + 0.2 * meta_weights.loc[row.name])
    val = np.random.poisson(lam)
    return int(np.clip(val, 0, 12))

print("Generating Feature 3: Protocol Amendment Count...")
df["protocol_amendment_count"] = df.apply(get_amendments, axis=1)

# ------------------------------------------------------------
# FEATURE 4: CRF Page Count (Normal Distributed)
# ------------------------------------------------------------
base_mean_phase = {"PHASE1": 45, "PHASE2": 85, "PHASE3": 140, "PHASE4": 60, "EARLY_PHASE1": 30, "NOT_REPORTED": 85}
interv_mult = {"BIOLOGICAL": 1.4, "DRUG": 1.2, "DEVICE": 1.1, "BEHAVIORAL": 0.6}

def get_crf_pages(row):
    phase = str(row["Phases"]).upper().strip()
    mean_val = base_mean_phase.get(phase, 85)
    
    interv = str(row["Intervention Type"]).upper().strip()
    mult = interv_mult.get(interv, 1.0)
    
    final_mean = mean_val * mult * (0.9 + 0.1 * meta_weights.loc[row.name])
    val = np.random.normal(loc=final_mean, scale=final_mean * 0.15)
    return int(max(10, round(val)))

print("Generating Feature 4: CRF Page Count...")
df["crf_page_count"] = df.apply(get_crf_pages, axis=1)

# ------------------------------------------------------------
# FEATURE 5: Site Experience Score (Uniform Distributed)
# ------------------------------------------------------------
ta_means = {
    "ONCOLOGY": 6.2, "CARDIOVASCULAR": 6.5, "INFECTIOUS DISEASE": 5.8,
    "METABOLIC": 6.0, "NEUROLOGY": 5.5, "PSYCHIATRY": 5.2,
    "RESPIRATORY": 6.3, "IMMUNOLOGY": 5.9, "OTHER": 5.5
}

def get_site_experience(row):
    ta = row["therapeutic_area"]
    target_mean = ta_means.get(ta, 5.5)
    
    # Higher sites/complexity metadata setup assumes stronger baseline center selection parameters
    modifier = 1.1 if row["Number of Sites"] > 15 else 0.95
    adjusted_mean = np.clip(target_mean * modifier, 1, 10)
    
    val = np.random.uniform(low=max(1, adjusted_mean - 2.0), high=min(10, adjusted_mean + 2.0))
    return round(val, 2)

print("Generating Feature 5: Site Experience Score...")
df["site_experience_score"] = df.apply(get_site_experience, axis=1)

# ------------------------------------------------------------
# FEATURE 6: Country Risk Score (Weighted Random Choice)
# ------------------------------------------------------------
def get_country_risk(row):
    num_sites = row["Number of Sites"]
    if num_sites > 25:
        p = [0.10, 0.20, 0.30, 0.25, 0.15] # Multi-country operations introduce complexity
    elif num_sites > 5:
        p = [0.20, 0.30, 0.30, 0.15, 0.05]
    else:
        p = [0.45, 0.35, 0.15, 0.04, 0.01] 
    return int(np.random.choice([1, 2, 3, 4, 5], p=p))

print("Generating Feature 6: Country Risk Score...")
df["country_risk_score"] = df.apply(get_country_risk, axis=1)

# ------------------------------------------------------------
# FEATURE 7: Investigator Tenure (Exponential Distributed)
# ------------------------------------------------------------
ta_tenure_mean = {"ONCOLOGY": 4.5, "CARDIOVASCULAR": 5.0, "NEUROLOGY": 4.0, "PSYCHIATRY": 3.8, "OTHER": 4.0}

def get_investigator_tenure(row):
    ta = row["therapeutic_area"]
    mean_val = ta_tenure_mean.get(ta, 4.0)
    
    # Tie tenure loosely to Phase architecture metrics (Late phases prefer established PIs)
    phase = str(row["Phases"]).upper().strip()
    mult = 1.3 if "PHASE3" in phase or "PHASE4" in phase else 0.9
    
    val = np.random.exponential(scale=mean_val * mult)
    return round(np.clip(val, 0.5, 20.0), 2)

print("Generating Feature 7: Investigator Tenure...")
df["investigator_tenure_years"] = df.apply(get_investigator_tenure, axis=1)

# ------------------------------------------------------------
# FEATURE 8: Screen Failure Rate (Beta Distributed)
# ------------------------------------------------------------
phase_beta_means = {"PHASE1": 0.22, "PHASE2": 0.32, "PHASE3": 0.26, "PHASE4": 0.15, "NOT_REPORTED": 0.25, "EARLY_PHASE1": 0.18}

def get_screen_failure(row):
    phase = str(row["Phases"]).upper().strip()
    target_mean = phase_beta_means.get(phase, 0.25)
    
    # Complex indications scale entry thresholds
    if row["therapeutic_area"] in ["ONCOLOGY", "NEUROLOGY"]:
        target_mean += 0.05
        
    target_mean = np.clip(target_mean, 0.05, 0.85)
    kappa = 12.0 
    alpha = target_mean * kappa
    beta_param = (1.0 - target_mean) * kappa
    
    return round(np.random.beta(alpha, beta_param), 4)

print("Generating Feature 8: Screen Failure Rate...")
df["screen_failure_rate"] = df.apply(get_screen_failure, axis=1)

# ------------------------------------------------------------
# FEATURE 9: Enrollment Velocity & 99th Percentile Clip (Fixes Issue 3)
# ------------------------------------------------------------
print("Calculating Feature 9: Enrollment Velocity (with outlier clipping)...")
df["enrollment_velocity"] = df["Enrollment"] / (df["Trial Duration (days)"] / 30.0)
df["enrollment_velocity"] = df["enrollment_velocity"].clip(lower=0.1)

# Clip extreme values at the 99th percentile to remove outliers
p99_value = df["enrollment_velocity"].quantile(0.99)
df["enrollment_velocity"] = df["enrollment_velocity"].clip(upper=p99_value)
print(f"  [INFO] Enrollment velocity clipped at 99th percentile value: {p99_value:.2f}/month")

# ------------------------------------------------------------
# FEATURE 10: Avg Time Visit to Data Entry (Gamma Distributed)
# ------------------------------------------------------------
def get_visit_to_entry(row):
    s_type = row["sponsor_type"]
    shape_param = 2.0
    scale_val = 1.5 if s_type == "INDUSTRY" else 3.0 # Academic institutions traditionally step slower
    
    val = np.random.gamma(shape=shape_param, scale=scale_val)
    return round(np.clip(val, 0.5, 30.0), 1)

print("Generating Feature 10: Visit to Data Entry Days...")
df["avg_time_visit_to_entry_days"] = df.apply(get_visit_to_entry, axis=1)

# ------------------------------------------------------------
# FEATURE 11: Edit Check Density Per Domain (Poisson Distributed)
# ------------------------------------------------------------
phase_lambda_checks = {"PHASE1": 12, "PHASE2": 22, "PHASE3": 38, "PHASE4": 18, "NOT_REPORTED": 22, "EARLY_PHASE1": 10}

def get_edit_checks(row):
    phase = str(row["Phases"]).upper().strip()
    base_lam = phase_lambda_checks.get(phase, 22)
    
    interv = str(row["Intervention Type"]).upper().strip()
    mult = interv_mult.get(interv, 1.0)
    
    lam = base_lam * mult * (0.85 + 0.15 * meta_weights.loc[row.name])
    return int(np.random.poisson(lam))

print("Generating Feature 11: Edit Check Density...")
df["edit_check_density_per_domain"] = df.apply(get_edit_checks, axis=1)

# ------------------------------------------------------------
# FEATURE 12: Historical Query Resolution Time (Gamma Distributed)
# ------------------------------------------------------------
def get_query_resolution_time(row):
    s_type = row["sponsor_type"]
    scale_val = 2.0 if s_type == "INDUSTRY" else 3.5
    
    val = np.random.gamma(shape=3.0, scale=scale_val)
    return round(np.clip(val, 1.0, 60.0), 1)

print("Generating Feature 12: Query Resolution Time...")
df["historical_query_resolution_time_days"] = df.apply(get_query_resolution_time, axis=1)

# ------------------------------------------------------------
# FEATURE 13: SDTM Domain Count (Poisson Distributed)
# ------------------------------------------------------------
phase_base_domains = {"PHASE1": 6, "PHASE2": 9, "PHASE3": 14, "PHASE4": 8, "NOT_REPORTED": 9, "EARLY_PHASE1": 5}

def get_sdtm_domains(row):
    phase = str(row["Phases"]).upper().strip()
    base_lam = phase_base_domains.get(phase, 9)
    
    interv = str(row["Intervention Type"]).upper().strip()
    mult = interv_mult.get(interv, 1.0)
    
    lam = base_lam * mult * (0.9 + 0.1 * meta_weights.loc[row.name])
    val = np.random.poisson(lam)
    return int(np.clip(val, 3, 20))

print("Generating Feature 13: SDTM Domain Count...")
df["sdtm_domain_count"] = df.apply(get_sdtm_domains, axis=1)

# ------------------------------------------------------------
# FEATURE 14: Complex Derivations Flag (Bernoulli Process)
# ------------------------------------------------------------
phase_prob_complex = {"PHASE1": 0.25, "PHASE2": 0.45, "PHASE3": 0.75, "PHASE4": 0.35, "NOT_REPORTED": 0.45, "EARLY_PHASE1": 0.15}

def get_complex_flag(row):
    phase = str(row["Phases"]).upper().strip()
    prob = phase_prob_complex.get(phase, 0.45)
    
    interv = str(row["Intervention Type"]).upper().strip()
    if interv in ["BIOLOGICAL", "GENETIC"]:
        prob += 0.15
        
    prob = np.clip(prob * (0.8 + 0.2 * meta_weights.loc[row.name]), 0.05, 0.95)
    return int(np.random.rand() < prob)

print("Generating Feature 14: Complex Derivations Flag...")
df["complex_derivations_flag"] = df.apply(get_complex_flag, axis=1)

# ------------------------------------------------------------
# FEATURE 15: External Data Feeds Count (Poisson Distributed)
# ------------------------------------------------------------
interv_feed_lambda = {"BIOLOGICAL": 3.2, "DRUG": 1.8, "DEVICE": 1.2, "BEHAVIORAL": 0.2}

def get_data_feeds(row):
    interv = str(row["Intervention Type"]).upper().strip()
    lam = interv_feed_lambda.get(interv, 1.0)
    
    val = np.random.poisson(lam)
    return int(np.clip(val, 0, 10))

print("Generating Feature 15: External Data Feeds Count...")
df["external_data_feeds_count"] = df.apply(get_data_feeds, axis=1)

# ============================================================
# FINAL POST-PIPELINE INTEGRITY AND SAVE TASKS
# ============================================================
print("\n" + "="*50 + "\nPIPELINE EXECUTION SANITY VERIFICATIONS\n" + "="*50)
print(f"Checking for any generated nulls: {df.isnull().sum().sum()}")
print(f"Final Data Structure Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

# Directional Correlation Validation Checklist
print("\nVerifying Natural (Leakage-Free) Spearman Correlations vs Target query_rate:")
for col in ["protocol_amendment_count", "crf_page_count", "country_risk_score", 
            "screen_failure_rate", "avg_time_visit_to_entry_days"]:
    corr = df[col].corr(df["query_rate"], method="spearman")
    print(f"  [VERIFICATION] Spearman correlation of {col:<35} vs query_rate: {corr:+.3f}")

df.to_csv(OUTPUT_FILE)
print(f"\nCompleted ML-Ready File Saved to: {OUTPUT_FILE}")

Dataset loaded successfully: 50,036 rows × 17 columns
Mapping Feature 1: Therapeutic Area (Broadened)...
Therapeutic Area Distribution Profile:
therapeutic_area
ONCOLOGY             81.82
OTHER                10.80
NEUROLOGY             2.99
INFECTIOUS DISEASE    1.22
METABOLIC             1.15
PSYCHIATRY            0.61
CARDIOVASCULAR        0.59
RESPIRATORY           0.42
IMMUNOLOGY            0.40
Name: proportion, dtype: float64

Mapping Feature 2: Sponsor Type...
Generating Feature 3: Protocol Amendment Count...
Generating Feature 4: CRF Page Count...
Generating Feature 5: Site Experience Score...
Generating Feature 6: Country Risk Score...
Generating Feature 7: Investigator Tenure...
Generating Feature 8: Screen Failure Rate...
Calculating Feature 9: Enrollment Velocity (with outlier clipping)...
  [INFO] Enrollment velocity clipped at 99th percentile value: 69.52/month
Generating Feature 10: Visit to Data Entry Days...
Generating Feature 11: Edit Check Density...
Generating Feat